<div align="center">
	<a href="https://dongwkim.com"><img src="logo.svg" alt="Logo" width="200" height="200"></a><br>
	<font size="20" style="font-family:IBM Plex Serif;font-variant: small-caps;">  Network Meta-Analysis</font><br>
</div>

</div>	
<br>
<hline></hline>
<font size = "1em">
<sup>1</sup> Department of Anatomy, Jagiellonian University, Kraków, Poland  <br>
<sup>2</sup> Whiting College of Engineering, Johns Hopkins University, Baltimore, MD, United States  <br>
<sup>3</sup> Harvard Dataverse, Harvard University, Cambridge, MA, United States
</font>

---

```mermaid
---
config:
  curve: stepBefore
---

graph LR

part1["systematic review"]
A["protocol"]
B["search strategy"]
C["search"]
D["deduplication"]
E["screening"]
E1["title and abstract<br>screening"]
E2["full-text screening"]

part2["data extraction"]
part3["meta-analysis"]

part1 --> A
A --> B
B --> C
C --> D
D --> E
E --> E1 & E2
E2 --> part2 --> part3
```

Define directory structure and store paths as global variables.

```
./research/
├── ./research/systematic_review/
|   |   
│   ├── ./research/systematic_review/protocol/  
|   |   |
│   │   ├── ./research/systematic_review/protocol/prospero  
│   │   └── ./research/systematic_review/protocol/cochrane  
|   |
│   ├── ./research/systematic_review/search_strategy/  
|   |   |
│   │   ├── ./research/systematic_review/search_strategy/pubmed  
│   │   ├── ./research/systematic_review/search_strategy/embase  
│   │   └── ./research/systematic_review/search_strategy/wos  
|   |
│   ├── ./research/systematic_review/search/  
|   |   |
│   │   ├── ./research/systematic_review/search/pubmed
│   │   ├── ./research/systematic_review/search/embase
│   │   └── ./research/systematic_review/search/wos
|   |
│   ├── ./research/systematic_review/deduplication/
|   |   |
│   │   ├── ./research/systematic_review/deduplication/doi
│   │   ├── ./research/systematic_review/deduplication/key
│   │   └── ./research/systematic_review/deduplication/id
|   |
│   └── ./research/systematic_review/screening/
|       |
│       ├── ./research/systematic_review/screening/title_abstract_screening
│       ├── ./research/systematic_review/screening/PDF
│       └── ./research/systematic_review/screening/full-text_screening
|
├── ./research/data
├── ./research/meta-analysis
├── ./research/manuscript
├── ./research/README.ipynb
├── ./research/docs
└── ./research/src
```

In [2]:
import os
import sys

root = os.getcwd()

systematic_review = r".//systematic_review"
protocol = r".//systematic_review//protocol"
search_strategy = r".//systematic_review//search_strategy"
search = r".//systematic_review//search"
deduplication = r".//systematic_review//deduplication"
screening = r".//systematic_review//screening"
title_abstract = r".//systematic_review//screening//title_abstract"
full_text = r".//systematic_review//screening//full-text"
pdf = r".//systematic_review//screening//PDF"

---

Import `subprocess`, `sys`, and `os` modules to activate a virtual environment in order to install PyPi packages without having to use terminal.

In [3]:
import subprocess
import sys
import os

# 1. Create virtual environment
subprocess.check_call([sys.executable, "-m", "venv", ".venv"])

# 2. Path to venv python (Windows)
venv_python = os.path.join(".venv", "Scripts", "python.exe")

# 3. Install requirements using venv python
subprocess.check_call([venv_python, "-m", "pip", "install", "-r", "requirements.txt"])

0

---

From the protocol that was developed, extract key terms from the eligibility criteria for inclusion and exclusion of studies in order to develop a *search strategy*.

In [40]:
# Firstly, the search strategies were written out for PubMed database

acl = f"""("anterior cruciate ligament"[mh] OR "anterior cruciate ligament reconstruction"[tiab])"""
rct = f"""("randomized controlled trial"[pt])"""
reviews = f"""("review"[pt] OR "systematic review"[pt] OR "meta-analysis"[pt]"""
outcomes = f"""(("ikdc"[tiab] OR "lysholm"[tiab] OR "tegner"[tiab] OR (("instrumental laxity"[tiab] OR "kt-1000"[tiab] OR "kt-2000"[tiab] OR "rolimeter"[tiab]) OR "pivot shift"[tiab] OR "lachman"[tiab]) OR ("graft failure"[tiab] OR "graft rupture"[tiab]))"""

bptb = f"""("bone-patellar tendon-bone"[tiab] OR "patellar tendon"[tiab] OR "bptb"[tiab])"""
ht = f"""("hamstring tendon"[tiab] OR "semitendinosus"[tiab] OR "gracilis"[tiab])"""
qt = f"""("quadriceps"[tiab] OR "qt"[tiab])"""
plt = f"""("peroneus longus"[tiab] OR "fibularis longus"[tiab])"""
at = f"""("achilles"[tiab])"""
ta = f"""("tibialis anterior"[tiab] OR "tibialis posterior"[tiab])"""

# Next, the subgroups were defined and key-value pairs were stored as a dictionary list.

subgroups = {"bptb": bptb, 
             "ht": ht, 
             "qt": qt, 
             "plt": plt, 
             "at": at, 
             "ta": ta}

# Using an iterative loop function, the search strategies were written as plain text files.

queries = {}

for x, y in subgroups.items():
    globals()[f"pm_{x}"] = f"{acl} AND {rct} AND {y}"
    with open(f"{search_strategy}/pubmed/pm_{x}.txt", 'w') as f:
        f.write(f"{acl} AND {rct} AND {y}")


# The search strategies were 'translated' via regular expressions from PubMed syntax to Embase and Web of Science syntax.

pm_queries = {
    "bptb": pm_bptb,
    "ht": pm_ht,
    "qt": pm_qt,
    "plt": pm_plt,
    "at": pm_at,
    "ta": pm_ta
}

# translate into embase queries from pubmed queries using regex and store them into the global environment.
import re

for x, y in pm_queries.items():
    query = re.sub(r"\[(.*?)\]",":\\1", y)
    query = re.sub(r"\:tiab",":ti,ab", query)
    query = re.sub(r"\:mh","/exp", query)
    query = re.sub(r"\:pt",":it,ti,ab", query)
    query = re.sub(r'"',"'", query)
    globals()[f"em_{x}"] = query
    with open(f"{search_strategy}/embase/em_{x}.txt", 'w') as f:
        f.write(query)

# translate into web of science queries from pubmed queries and stores them into the global environment
for x, y in pm_queries.items():
    query = re.sub(r'"(.*?)"\[(.*?)\]','\\2="\\1"', y)
    query = re.sub(r'tiab="(.*?)"','(TI=(\\1) OR AB=(\\1))', query)
    query = re.sub(r'mh="(.*?)"','(TMIC=(\\1))', query)
    query = re.sub(r'pt="(.*?)"','(TS=(\\1))', query)
    globals()[f"wos_{x}"] = query

    with open(f"{search_strategy}/wos/wos_{x}.txt", 'w') as f:
        f.write(query)

# So here, what we did was: write out the search strategies for PubMed, and they were written and saved as plain text files in not only PubMed syntax, but in Embase and Web of Science as well. 

---

A script to either create a search strategy using the terms, field tags, and Boolean operators and save them as plain text files or load already written and saved plain text files for import into the API search scripts. This uses the search strategies (e.g., `pm_bptb.txt`, search strategy in plain text written in PubMed syntax for bone-patellar tendon-bone (BPTB) subgroup search) and pulls data from PubMed to output PMIDs (`pmid_pm_bptb.txt`) and search results in XML (`pm_bptb.xml`) and parses this into CSV files (`pm_bptb.csv`).

In [41]:
# And here's a script to even walk you through the creation of search strategies themselves.
# Also, this script uses these search strategies to pull the data from the search directly from PubMed and save them as CSV files.
# CHANGELOG: place pmids and XMLs in separate folders, or don't write them at all, maybe.

import pandas as pd
from Bio import Entrez, Medline
import ssl
import certifi

ssl._create_default_https_context = lambda: ssl.create_default_context(
    cafile=certifi.where()
)
# create search strategy using structured inputs

question = input("Do you already have a search strategy file saved?")
filename = input("Enter the file name of the search strategy: ")
file = f"{search_strategy}/pubmed/{filename}.txt"

if question == "no":
    parts = []
    string = []
    while True:
        term = input("Enter the search string: ")
        field = input("Enter the field type: ")
        string.append(f"'{term}'[{field}]")
        boolean = input("Enter the Boolean operator (e.g., OR, AND, NOT): ")
        
        if boolean == "OR":
            continue
    
        parts.append("(" + " OR ".join(string) + ")")
        string = []
    
        if boolean == "":
            break
            
        parts.append(boolean)
        
    query = " ".join(parts) # query = search strategy FROM HERE
    with open(file, "w") as f:
        f.write(query)
    
with open(file, "r") as f:
    query = f.read()

query = f"{query}"

# use NCBI's e-utitilies to pull PMIDs using e-search.

Entrez.email = "dkim246@jhmi.edu"
Entrez.api_key = 'bb1c481d8e167acd16f3616593c18b3aab08'

handle = Entrez.esearch(db= "pubmed", 
                        term = query, 
                        usehistory = "y", 
                        retmax = 1000,
                        retmode = "xml")

pmid = Entrez.read(handle)

pmid = pmid['IdList']
pmid = ",".join(pmid) # list to string
#with open(f"./data/pmid_{filename}.txt", 'w') as f:
#    f.write(pmid)
with open(f"{search}/pubmed/pmid_{filename}.txt", 'w') as f:
    f.write(pmid)
handle.close()

# ncbi e-summary
handle = Entrez.esummary(db= "pubmed", 
                         id = pmid, 
                         retmode = "xml", 
                         usehistory = "y", 
                         retmax = 1000)

xml = handle.read()
#xml_file = f"./data/{filename}.xml"
xml_file = f"{search}/pubmed/{filename}.xml"
with open(xml_file, "wb") as f:
    f.write(xml)   
handle.close()

import xml.etree.ElementTree as ET

tree = ET.parse(f"{xml_file}")
root = tree.getroot()

docsum = root[0]

def xml_parse(docsum):
    df = {}
    df["pmid"] = docsum.find("Id").text
    for item in docsum.findall("Item"):
        key = item.attrib.get("Name")
        if item.attrib.get("Type") == "List":
            values = [sub.text for sub in item.findall("Item") if sub.text]
            df[key] = values
        else:
            df[key] = item.text
    return df
records = [xml_parse(doc) for doc in root.findall(".//DocSum")]
df = pd.DataFrame(records)

csv_file = f"{search}/pubmed/{filename}.csv"
df.to_csv(csv_file, encoding = "utf-8")

# using e-fetch, the abstracts are pulled

handle = Entrez.efetch(
    db="pubmed",
    id=pmid,
    rettype="medline",
    retmode="text"
)

text = list(Medline.parse(handle))
handle.close()

abstracts = pd.DataFrame(text)[["PMID", "AB"]]
abstracts.rename(columns = {"PMID":"pmid", "AB":"abstract"}, inplace = True)
df = df.merge(abstracts, on = "pmid", how = "left")
csv_file = f"{search}/pubmed/{filename}.csv"
df.to_csv(csv_file, encoding = "utf-8")

Do you already have a search strategy file saved? no
Enter the file name of the search strategy:  test
Enter the search string:  
Enter the field type:  
Enter the Boolean operator (e.g., OR, AND, NOT):  


KeyError: "None of [Index(['PMID', 'AB'], dtype='str')] are in the [columns]"

In [18]:
# convert Web of Science exports to csv

import pandas as pd

subgroups = {
    "bptb": "patellar",
    "ht": "hamstring",
    "qt": "quadriceps",
    "plt": "peroneus",
    "at": "achilles",
    "ta": "tibialis"}

for x, y in subgroups.items():
    df = pd.read_csv(f"{search}/wos/tsv/wos_{x}.tsv", sep = '\t', encoding = "latin-1")
    df.to_csv(f"{search}/wos/wos_{x}.csv", encoding = "utf-8", sep = ",", index = False)
    globals()[f"wos_{x}"] = df

The CSV files from the three databases were now cleaned, prepared, and transformed; this is otherwise known as 'data wrangling' in Data Science.

In [29]:
# import the exported csv files into the global environment
import pandas as pd 
search = f"./systematic_review/search"

databases = {"pm": "pubmed", 
             "em": "embase", 
             "wos": "wos"}

subgroups = {"bptb": "patellar", 
             "ht": "hamstring", 
             "qt": "quadriceps", 
             "plt": "peroneus", 
             "at": "achilles", 
             "ta": "tibialis"}

# reduced from 45 lines of code to 12
for a, b in databases.items():
    dfs = []
    for x, y in subgroups.items():
        df = pd.read_csv(f"{search}/{b}/{a}_{x}.csv", encoding = "utf-8")
        df['source'] = b
        df['subgroup'] = x
        dfs.append(df)
        globals()[f"{a}_{x}"] = df
        data = pd.concat(dfs, ignore_index = True)
        data.insert(0, "id", range(1, len(data) + 1))
        data.to_csv(f"{search}/{b}/{b}.csv", encoding = "utf-8", index = False)
        globals()[f"{b}"] = data

In [35]:
# info() to get column names in order to match them up so that one unified 'records.csv' file can be created.

pubmed = pd.DataFrame(pubmed)
embase = pd.DataFrame(embase)
wos = pd.DataFrame(wos)

pubmed = pubmed.rename(columns = {
        'PMID': 'pmid', 
        'DOI': 'doi',
        'Authors': 'authors',
        'Title': 'title',
        'Publication Year': 'year',
        'Journal/Book': 'journal'}, inplace = True)

embase = embase.rename(columns = {
        'Embase Link': 'id',
        'DOI':'doi',
        'Author Names': 'authors',
        'Title': 'title',
        'Publication Year': 'year',
        'Source': 'journal'
    }, inplace = True)

wos = wos.rename(columns = {    
    'UT':'id', 
    'DOI':'doi',
    'AU': 'authors',
    'TI':  'title',
    'PY':  'year',
    'JI': 'journal',
}, inplace = True)


#pubmed = pd.read_csv(f"{search}/pubmed/pubmed.csv", encoding = "utf-8", usecols = ["id", "source", "subgroup", "pmid", "doi", "title", "authors", "year", "journal", "abstract"])



records = pd.concat([pubmed, embase, wos])


ValueError: All objects passed were None

In [33]:
records = pd.DataFrame({
        "source": recorfsource, 
        "subgroup":records.subgroup, 
        "pmid":    records.pmid,
        "doi":     records.doi, 
        "title":   records.title, 
        "authors": records.authors, 
        "year":    records.year, 
        "journal": records.journal, 
        "abstract":records.abstract
    })


index = range(0, len(records)+1)
index.records = index

ValueError: All objects passed were None

deduplication

In [ ]:
import pandas as pd
import mermaid

input_file_name = input('Enter file name: ') + '.csv'
df = pd.read_csv(input_file_name) # A (records)
cols_input = input('Enter the columns for which to deduplicate based on: ')
cols = [c.strip() for c in cols_input.split(',')]
output_file_name = './' + '_'.join(cols) + '_deduplicated.csv'
prisma_file_name = output_file_name.replace('.csv', '.mmd')

nulls_mask = df[cols].isnull().any(axis=1)
df_nulls = df[nulls_mask] # B
df_non_nulls = df[~nulls_mask] # C

duplicates_mask = df_non_nulls.duplicated(subset = cols, keep = False)
df_non_duplicates = df_non_nulls[~duplicates_mask] # D
df_duplicates = df_non_nulls[duplicates_mask] # E
df_kept = df_duplicates.drop_duplicates(subset = cols, keep = 'first')
df_removed = df_duplicates[~df_duplicates.index.isin(df_kept.index)]
df_unique = df_non_nulls.drop_duplicates(subset = cols, keep = 'first') # df of unique
df_deduplicated = pd.concat([df_non_duplicates, df_kept], ignore_index=True) # df of unique + df of non-duplicates

results = {"records": len(df),  
"nulls": len(df_nulls), 
"non_nulls": len(df_non_nulls), 
"non_duplicates": len(df_non_duplicates), 
"duplicates": len(df_duplicates), 
"removed": len(df_removed), 
"kept": len(df_kept),
"unique": len(df_unique),
"deduplicated": len(df_deduplicated)
}

df_nulls.to_csv(output_file_name.replace('deduplicated','nulls'), index = False)
df_deduplicated.to_csv(output_file_name, index = False)
df_removed.to_csv(output_file_name.replace('.csv', '_removed.csv'), index = False)

graph_text = f"""---
config:
theme: neutral
curve: stepBefore
---
graph TD;
A["**records** (*n* = {results['records']})"];
B["null (*n* = {results['nulls']})"];
C["non-null (*n* = {results['non_nulls']})"];
D["non-duplicates (*n* = {results['non_duplicates']})"];
E["duplicates (*n* = {results['duplicates']})"];
F["duplicates kept (*n* = {results['kept']})"];
G["duplicates removed (*n* = {results['removed']})"];
H["unique (*n* = {results['unique']})"];
I["deduplicated (*n* = {results['deduplicated']})"];

A --> B & C;
C --> D & E;
E --> F & G;
D & F --> H"""


with open(prisma_file_name, "w") as f:
    f.write(graph_text)

screening